# Training Belief Distribution — Target Type Branch

Loads pre-trained FFNN, LSTM, and SVM models, runs inference on the
**training set**, fuses the resulting belief distributions, and saves
`train_belief_distribution.csv` — which is consumed by `main-intent.ipynb`
as the membership input for the intent branch.

All tuneable values live in **`params.py`** — edit there, not here.

## 1. Imports

In [13]:
%load_ext autoreload
%autoreload 2

from src.config import *
from src.preprocess import *
from src.model_training.models import *
from src.postprocess import *

import params   # central configuration — see params.py

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Load Dataset

In [14]:
data = pd.read_csv(params.INPUT_PATH, header=None)

# Features: columns 0 – 8999  |  Label: second-to-last column
X = data.iloc[:, 0:9000]
y = data.iloc[:, -2]

print(f"Dataset loaded — X: {X.shape}, y: {y.shape}")
print(f"Unique classes : {sorted(y.unique())}")

Dataset loaded — X: (4000, 9000), y: (4000,)
Unique classes : ['Bomber', 'CivilianCommercialAirliner', 'CivilianHelicopter', 'CruiseMissile', 'MilitaryHelicopter', 'MultiroleFighter', 'SmallCivilian', 'SmallUnmanned']


## 3. Train / Test Split

In [15]:
train_concat, test_concat = split_dataset(
    X, y,
    test_size=params.TYPE_TEST_SIZE,
)

print(f"Train : {len(train_concat):>6,} samples")
print(f"Test  : {len(test_concat):>6,} samples")

Train :  3,200 samples
Test  :    800 samples


## 4. Load Pre-trained Models

In [16]:
import joblib
from tensorflow.keras.models import load_model

# ── Preprocessing artefacts ────────────────────────────────────────────────
trans_standard = joblib.load(params.SCALER_TRANSFORM_PATH)
stat_standard  = joblib.load(params.SCALER_STAT_PATH)
encoding       = joblib.load(params.OHE_PATH)

# ── Classification models ──────────────────────────────────────────────────
ffnn_model = load_model(params.FFNN_MODEL_PATH)
lstm_model = load_model(params.LSTM_MODEL_PATH)
svm_model  = joblib.load(params.SVM_MODEL_PATH)

print("All models loaded.")

All models loaded.


## 5. Preprocessing — Training Set

In [17]:
# ── Sliding-window slicing ─────────────────────────────────────────────────
train_slices = slice_dataset(train_concat, params.K1, params.M, params.W)

# ── Statistical Feature Extraction ────────────────────────────────────────
train_stat = SFE_dataset(train_slices, params.K1, params.W)

# ── Trajectory Transformation ──────────────────────────────────────────────
train_transform = transform_dataset(train_slices, params.K1, params.W)

# ── Label encoding ─────────────────────────────────────────────────────────
y_train_encoded = encoding.transform(
    train_stat.iloc[:, -1].values.reshape(-1, 1)
)

# ── Feature scaling ────────────────────────────────────────────────────────
train_stat_scaled      = stat_standard.transform(pd.DataFrame(train_stat).iloc[:, :-1])
train_transform_scaled = trans_standard.transform(pd.DataFrame(train_transform).iloc[:, :-1])

print(f"Stat features shape      : {train_stat_scaled.shape}")
print(f"Transform features shape : {train_transform_scaled.shape}")

Stat features shape      : (64000, 99)
Transform features shape : (64000, 450)


## 6. FFNN — Training Set Inference

In [18]:
ffnn_prob_distribution = ffnn_model.predict(train_stat_scaled, verbose=0)

loss, ffnn_accuracy = ffnn_model.evaluate(
    train_stat_scaled, y_train_encoded, verbose=0
)

#print(f"FFNN Accuracy : {ffnn_accuracy:.6f}")

## 7. LSTM — Training Set Inference

In [19]:
# LSTM requires 3-D input: samples × window × features
train_lstm_input = train_transform_scaled.reshape(
    train_transform_scaled.shape[0], params.W, params.K1
)

lstm_prob_distribution = lstm_model.predict(train_lstm_input, verbose=0)

loss, lstm_accuracy = lstm_model.evaluate(
    train_lstm_input, y_train_encoded, verbose=0
)

#print(f"LSTM Accuracy : {lstm_accuracy:.6f}")

## 8. SVM — Training Set Inference

In [20]:
y_train = train_stat.iloc[:, -1]

svm_pred             = svm_model.predict(train_stat_scaled)
svm_accuracy         = metrics.accuracy_score(y_train, svm_pred)
svm_prob_distribution = svm_model.predict_proba(train_stat_scaled)

#print(f"SVM Accuracy  : {svm_accuracy:.6f}")

## 10. Belief Mapping & Classifier Fusion

In [21]:
# ── Belief mapping ─────────────────────────────────────────────────────────
svm_belief, ffnn_belief, lstm_belief = belief_mapping_type(
    svm_prob_distribution,  svm_accuracy,
    ffnn_prob_distribution, ffnn_accuracy,
    lstm_prob_distribution, lstm_accuracy,
)

# ── Classifier fusion: FFNN ⊕ LSTM → then ⊕ SVM ──────────────────────────
fused_ffnn_lstm     = fuse_ffnn_lstm(ffnn_belief, lstm_belief)
fused_ffnn_lstm_svm = fuse_ffnn_lstm_svm(fused_ffnn_lstm, svm_belief)

print("Belief mapping and classifier fusion complete.")

Belief mapping and classifier fusion complete.


## 11. Temporal Fusion

In [22]:
final_fused = pd.concat(
    [
        pd.DataFrame(temporal_fusion_type(fused_ffnn_lstm_svm)),
        train_stat.iloc[:, -1].reset_index(drop=True),
    ],
    axis=1,
)

print(f"Final fused output shape: {final_fused.shape}")
final_fused.head()

Final fused output shape: (64000, 12)


,0,1,2,3,4,5,6,7,8,9,10,Label
0,7.963540e-06,5.963680e-06,1.081169e-05,0.984984,2.914773e-05,0.000444,5.730201e-05,0.000049,4.878734e-05,4.865170e-05,0.014314,CruiseMissile
1,1.696796e-07,9.017687e-08,1.663756e-07,0.999781,4.568127e-07,0.000011,8.370049e-07,0.000001,8.158884e-07,8.211818e-07,0.000203,CruiseMissile
2,1.740170e-07,9.121856e-08,1.742235e-07,0.999776,4.842266e-07,0.000015,8.551128e-07,0.000001,8.505265e-07,8.581223e-07,0.000205,CruiseMissile
3,1.757509e-07,9.014744e-08,1.784518e-07,0.999775,4.969149e-07,0.000015,8.586631e-07,0.000001,8.581534e-07,8.765369e-07,0.000205,CruiseMissile
4,1.854794e-07,9.262423e-08,1.819271e-07,0.999772,4.936407e-07,0.000017,8.605505e-07,0.000002,8.829888e-07,8.551880e-07,0.000206,CruiseMissile


## 12. Save Output

In [23]:
import os
os.makedirs(params.TYPE_OUTPUT_DIR, exist_ok=True)

final_fused.to_csv(
    params.TYPE_TRAIN_OUTPUT_CSV,
    index=False,
    header=None,
)

print(f"Saved → {params.TYPE_TRAIN_OUTPUT_CSV}")

Saved → C:\Users\tenzi\Desktop\MS_Research_Work\Combat Identification\output\target_type_belief_distribution\train_belief_distribution.csv
